# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmAjitJadhav/flyrank-ml-internship-om/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research question

Which content items show meaningful recent search-performance decline or recovery, and which declining pages should a search team prioritize for refresh or manual review?

### Decision supported

The goal is to turn recent FlyRank search-performance data into an interpretable content-opportunity queue. The system identifies whether content is growing, declining, recovering, or worth review, then prioritizes declining and ambiguous items by estimated impressions at stake and provides reason codes for review.

This is a decision-support and prioritization system, not a causal model of Google's ranking algorithm.

In [1]:
import pandas as pd
import numpy as np
QUESTION = """
Which content items show meaningful recent search-performance decline or recovery,
and which declining pages should a search team prioritize for refresh or manual review?
"""

DECISION = """
Prioritize content refresh and manual-review work using observed search-performance
changes, with explanations and estimated impressions at stake.
"""

print("Research question:", QUESTION.strip())
print("Decision supported:", DECISION.strip())

Research question: Which content items show meaningful recent search-performance decline or recovery,
and which declining pages should a search team prioritize for refresh or manual review?
Decision supported: Prioritize content refresh and manual-review work using observed search-performance
changes, with explanations and estimated impressions at stake.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data source

This analysis uses the public-safe FlyRank internship warehouse release on Hugging Face.

The main source is the daily content-performance table. Query-level aggregates are used as supporting signals.

### Analysis window

The analysis uses three consecutive 30-day windows ending at the stable cutoff of 2026-06-25:

- last30: most recent 30 days
- prev30: 30–60 days before the cutoff
- older30: 60–90 days before the cutoff

A stable cutoff was used because the most recent trailing days showed a change in daily content coverage.

### Inclusion rule

Content items were retained only when prior and older windows contained at least 100 impressions, so percentage changes were not dominated by extremely small denominators.

The final feature table contains 91,587 content/client observations.

No client names, domains, private queries, credentials, or raw private exports are included in the paper outputs.

In [2]:
STABLE_CUTOFF = "2026-06-25"
MIN_PRIOR_IMPRESSIONS = 100

DATA_SUMMARY = {
    "source": "FlyRank/internship-warehouse",
    "daily_table": "fact_content_daily_performance",
    "query_table": "fact_content_query_90d",
    "stable_cutoff": STABLE_CUTOFF,
    "window_days": 30,
    "minimum_prior_impressions": MIN_PRIOR_IMPRESSIONS,
    "final_feature_rows": 91587,
}

pd.Series(DATA_SUMMARY)

,0
source,FlyRank/internship-warehouse
daily_table,fact_content_daily_performance
query_table,fact_content_query_90d
stable_cutoff,2026-06-25
window_days,30
minimum_prior_impressions,100
final_feature_rows,91587


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Method

I use a rule-based content-opportunity scoring approach built from three consecutive 30-day windows.

For each content item, I calculate impressions, clicks, average search position, and CTR for:

- `last30`: the most recent 30 days
- `prev30`: 30–60 days before the cutoff
- `older30`: 60–90 days before the cutoff

The three-window trajectory allows the system to distinguish continued decline from recovery.

### State definitions

Each content item is assigned one of four states:

- `growing`: recent performance is improving.
- `declining`: recent performance is continuing to deteriorate.
- `recovering`: performance declined previously but has improved in the latest window.
- `worth_review`: signals are ambiguous, conflicting, or otherwise require manual review.

A minimum prior-impression threshold of 100 is used to avoid unstable percentage changes from extremely small traffic bases.

### Reason codes

Declining and review items receive interpretable reason codes based on observed signals:

- worsening average position
- high dependence on a single query
- high impression volatility
- CTR decline despite relatively stable impressions
- severe broad decline

A coefficient-of-variation feature for daily impressions is used as the additional engineered volatility signal.

### Validation

Validation uses a client-grouped split with `GroupShuffleSplit` rather than a random row split. This reduces the risk that rows from the same client appear in both groups and makes the validation more representative of cross-client generalization.

The analysis is observational and rule-based. It is intended to support prioritization decisions, not to establish causal effects or prove how Google's ranking system works.

In [3]:
# Methodology parameters used in the final run

METHODOLOGY = {
    "stable_cutoff": "2026-06-25",
    "window_length_days": 30,
    "history_windows": 3,
    "minimum_prior_impressions": 100,
    "position_worsening_threshold": 1.0,
    "top_query_share_threshold": 0.70,
    "high_volatility_cv_threshold": 1.0,
    "validation": "GroupShuffleSplit by client_hash_id",
    "approach": "rule-based decision-support scoring",
}

pd.Series(METHODOLOGY)

,0
stable_cutoff,2026-06-25
window_length_days,30
history_windows,3
minimum_prior_impressions,100
position_worsening_threshold,1.0
top_query_share_threshold,0.7
high_volatility_cv_threshold,1.0
validation,GroupShuffleSplit by client_hash_id
approach,rule-based decision-support scoring


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Results

This capstone is a rule-based decision-support system rather than a supervised predictive model, so no predictive accuracy claim is made.

The baseline is a simple two-window rule: classify an item as declining when recent 30-day impressions are at least 15% below the preceding 30 days. The final system adds a third 30-day window, distinguishes growing, declining, recovering, and worth-review states, and attaches interpretable reason codes.

The final analysis contains 91,587 content/client observations:

| Measure | Final system |
|---|---:|
| Declining | 44.89% |
| Worth review | 30.89% |
| Recovering | 14.57% |
| Growing | 9.66% |

The final action queue contains the declining and worth-review items and ranks them by estimated impressions at stake:

`max(imp_prev30 - imp_last30, 0)`

The highest-priority items therefore combine meaningful performance loss with meaningful traffic exposure, rather than being ranked by percentage change alone.

These results are observational and directional. They show where the scoring system identifies potential content opportunities; they do not demonstrate that refreshing a page will cause rankings, impressions, or clicks to recover.

In [4]:
import pandas as pd

data = pd.read_csv("content_opportunity_scores_full.csv")

print("Loaded rows:", len(data))
print(data.columns.tolist())

baseline_declining = data['chg_recent'] <= -0.15

results = pd.DataFrame({
    "metric": [
        "Final feature rows",
        "Baseline declining share",
        "Final declining share",
        "Final worth-review share",
        "Final recovering share",
        "Final growing share",
        "Final actionable rows",
    ],
    "value": [
        len(data),
        baseline_declining.mean(),
        (data['state'] == 'declining').mean(),
        (data['state'] == 'worth_review').mean(),
        (data['state'] == 'recovering').mean(),
        (data['state'] == 'growing').mean(),
        data['state'].isin(['declining', 'worth_review']).sum(),
    ]
})

display(results)

print("\nBaseline = two-window rule: chg_recent <= -0.15")
print("Final system = three-window state classification + reason codes")

FileNotFoundError: [Errno 2] No such file or directory: 'content_opportunity_scores_full.csv'

## 5. Limitations

*What this work cannot claim.*

### Limitations and honest framing

This analysis is observational and intended for decision support. The rules identify pages whose measured search-performance signals changed; they do not establish that those changes were caused by content quality, a search-engine update, or any other single factor.

The analysis uses a stable cutoff of 2026-06-25 because daily content coverage dropped in the trailing days after that point. This means the results should be interpreted as a directional snapshot for the selected analysis window, not as a real-time measurement.

The results are also highly concentrated by client. The largest clients account for a large share of the analyzed content, so aggregate state proportions may reflect client mix as well as broader search-performance patterns.

Query-mix signals are not available for every content item, so some reason codes cannot be assigned for all rows.

Finally, the opportunity ranking estimates impressions at stake from observed historical differences. It should be used to prioritize investigation and refresh work, not as a forecast of guaranteed traffic recovery.

In [ ]:
LIMITATIONS = {
    "analysis_type": "observational, rule-based decision support",
    "stable_cutoff": "2026-06-25",
    "causal_claim": False,
    "real_time_claim": False,
    "query_mix_coverage": "approximately 94%",
    "primary_ranking_metric": "historical impressions at stake",
}

pd.Series(LIMITATIONS)

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Ranked recommendations

The action queue prioritizes content using historical impressions at stake:

`max(imp_prev30 - imp_last30, 0)`

This avoids prioritizing a page only because its percentage decline is large. A smaller percentage decline on a high-impression page can represent a larger practical opportunity.

The recommended actions are:

- `refresh_or_review` for items classified as declining.
- `manual_review` for items classified as worth_review.
- `monitor` for recovering items.
- `protect_do_not_touch` for growing items.

The ranking is intended to help a search/content team decide what to investigate first. It is not a prediction of guaranteed traffic recovery.

In [ ]:
# Load the final ranked action list produced by ML-CAP-01.

ranked = pd.read_csv("content_opportunity_ranked_actions.csv")

print("Ranked rows:", len(ranked))

display(
    ranked[
        [
            "content_hash_id",
            "client_hash_id",
            "state",
            "reason_code",
            "recommended_action",
            "imp_prev30",
            "imp_last30",
            "impressions_at_stake",
            "chg_recent",
        ]
    ].head(25)
)

In [ ]:
print("Action counts:")
print(ranked["recommended_action"].value_counts())

print("\nTop 10 by impressions at stake:")
display(
    ranked.sort_values(
        "impressions_at_stake",
        ascending=False
    ).head(10)
)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

### Paper artifacts

The deployed paper will present:

1. The final state distribution across the 91,587 analyzed content/client observations.
2. A ranked-action summary showing the highest-priority content opportunities by impressions at stake.
3. A reason-code breakdown showing the most common observed signals associated with declining or review-worthy content.
4. The client-group validation result used to assess whether the state distribution is broadly consistent across client groups.

The public paper will use aggregated results and will not expose client names, domains, URLs, private queries, credentials, or raw warehouse exports.

In [ ]:
# Prepare aggregated, public-safe artifacts for the paper.

state_summary = (
    data["state"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename_axis("state")
    .reset_index(name="share_percent")
)

reason_summary = (
    ranked["reason_code"]
    .value_counts()
    .head(10)
    .rename_axis("reason_code")
    .reset_index(name="count")
)

top_opportunities = (
    ranked[
        [
            "state",
            "reason_code",
            "recommended_action",
            "imp_prev30",
            "imp_last30",
            "impressions_at_stake",
            "chg_recent",
        ]
    ]
    .head(10)
)

print("State summary:")
display(state_summary)

print("Reason-code summary:")
display(reason_summary)

print("Top opportunities:")
display(top_opportunities)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
